# The optimized version for reducing RAM usage
**This is based on the work of [jjinho](https://www.kaggle.com/jjinho) from [his notebook](https://www.kaggle.com/code/jjinho/open-book-llm-science-exam)
Thank you jjinho so much for his great work.
I try to reduce the memory leak. At the moment, after getting the train_context.csv, the RAM usage is around 4~5 GB (GPU usage is 1 GB). Hope anyone can make a futher improvement**

# Kaggle Large Language Model Science Exam

In this competition we are challenged to answer difficult science-based questions written by a Large Language Model. We are also told that 

> The dataset for this challenge was generated by giving gpt3.5 snippets of text on a range of scientific topics pulled from wikipedia, and asking it to write a multiple choice question (with a known answer), then filtering out easy questions.

An idea is to make this challenge a little easier by converting it to an ***open book science exam*** using semantic search and Wikipedia.

## Overview

1. We obtain the plain text version of the latest dump from Wikipedia (https://www.kaggle.com/datasets/jjinho/wikipedia-20230701)
1. We will then convert the prompts into embeddings using sentence transformers (specifically using the `all-MiniLM-L6-v2` model)
1. We will also create embeddings of all the Wikipedia articles, and to help us, use the first sentence from each article to provide more context (again using `all-MiniLM-L6-v2`)
1. We will then use `faiss` to perform similarity search to find the top-k articles that are most likely to have the information needed
1. We will then get the full text of those articles and split them into sentences using the fast `blingfire` package
1. Again, we will obtain embeddings of these sentences as well as embeddings of the prompt + answer choices and perform similarity search to get the top-k matching sentences for each question
1. We can then combine the questions, answer choices, and context to either perform straight up question answering, or feed into a LLM

## TODO

* ~~Enable off-line use~~
* Improve memory efficiency
* Make faster
* Use context information to train a model or run inference using LLM (like https://www.kaggle.com/code/philippsinger/h2ogpt-perplexity-ranking/notebook)

# Get Necessary Packages

In [1]:
!pip install -U /kaggle/input/faiss-cpu-173/faiss_cpu-1.7.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl

Processing /kaggle/input/faiss-cpu-173/faiss_cpu-1.7.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl


In [2]:
## Needed otherwise encounter read-only error
!cp -rf /kaggle/input/sentence-transformers-222/sentence-transformers /kaggle/working/sentence-transformers

In [3]:
!cp /kaggle/input/datasets-wheel/datasets-2.14.4-py3-none-any.whl /kaggle/working
!pip install  /kaggle/working/datasets-2.14.4-py3-none-any.whl

Processing ./datasets-2.14.4-py3-none-any.whl
  Attempting uninstall: datasets
    Found existing installation: datasets 2.1.0
    Uninstalling datasets-2.1.0:
      Successfully uninstalled datasets-2.1.0


In [4]:
!pip install -U /kaggle/working/sentence-transformers

Processing ./sentence-transformers
  Preparing metadata (setup.py) ... - \ done
  Created wheel for sentence-transformers: filename=sentence_transformers-2.2.2-py3-none-any.whl size=126134 sha256=1980cd2733519309018206e6385d5f83e4bda61ae332634c2a6e83ae87f73f54
  Stored in directory: /root/.cache/pip/wheels/6c/ea/76/d9a930b223b1d3d5d6aff69458725316b0fe205b854faf1812
Successfully built sentence-transformers


In [5]:
# installing offline dependencies
!pip install -U /kaggle/input/faiss-gpu-173-python310/faiss_gpu-1.7.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
!cp -rf /kaggle/input/sentence-transformers-222/sentence-transformers /kaggle/working/sentence-transformers
!pip install -U /kaggle/working/sentence-transformers
!pip install -U /kaggle/input/blingfire-018/blingfire-0.1.8-py3-none-any.whl

!pip install --no-index --no-deps /kaggle/input/llm-whls/transformers-4.31.0-py3-none-any.whl
!pip install --no-index --no-deps /kaggle/input/llm-whls/peft-0.4.0-py3-none-any.whl
!pip install --no-index --no-deps /kaggle/input/llm-whls/trl-0.5.0-py3-none-any.whl

Processing /kaggle/input/faiss-gpu-173-python310/faiss_gpu-1.7.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: '/kaggle/input/faiss-gpu-173-python310/faiss_gpu-1.7.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'

Processing ./sentence-transformers
  Preparing metadata (setup.py) ... - \ done
  Created wheel for sentence-transformers: filename=sentence_transformers-2.2.2-py3-none-any.whl size=126134 sha256=221f33009e59f7aa6c91514c0c21cbf207a9247fc5197c4bfc8a62f9a7cd00e1
  Stored in directory: /root/.cache/pip/wheels/6c/ea/76/d9a930b223b1d3d5d6aff69458725316b0fe205b854faf1812
Successfully built sentence-transformers
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 2.2.2
    Uninstalling sentence-transformers-2.2.2:
      Successfully uninstalled sentence-transformers-2.2.2
Processing /kaggle/input/blingfire-018/blingf

In [6]:
!pip install -U /kaggle/input/blingfire-018/blingfire-0.1.8-py3-none-any.whl

Processing /kaggle/input/blingfire-018/blingfire-0.1.8-py3-none-any.whl
blingfire is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.


## Imports

In [7]:
import os
import gc
import pandas as pd
import numpy as np
import re
from tqdm.auto import tqdm
import blingfire as bf

from collections.abc import Iterable

import faiss
from faiss import write_index, read_index

from sentence_transformers import SentenceTransformer

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [8]:
!cp -r /kaggle/input/stem-wiki-cohere-no-emb /kaggle/working
!cp -r /kaggle/input/all-paraphs-parsed-expanded /kaggle/working/

In [9]:
import torch
import os
import pandas as pd
import ctypes
import numpy as np
import pandas as pd 
from datasets import load_dataset, load_from_disk
from sklearn.feature_extraction.text import TfidfVectorizer
import torch
from transformers import LongformerTokenizer, LongformerForMultipleChoice
import transformers
import pandas as pd
import pickle
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import unicodedata

from datasets import load_dataset, load_from_disk

libc = ctypes.CDLL("libc.so.6")

/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:98: UserWarning: unable to load libtensorflow_io_plugins.so: unable to open file: libtensorflow_io_plugins.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so: undefined symbol: _ZN3tsl6StatusC1EN10tensorflow5error4CodeESt17basic_string_viewIcSt11char_traitsIcEENS_14SourceLocationE']
  warnings.warn(f"unable to load libtensorflow_io_plugins.so: {e}")
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:104: UserWarning: file system plugins are not loaded: unable to open file: libtensorflow_io.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io.so: undefined symbol: _ZTVN10tenso

In [10]:
# train = pd.read_csv("/kaggle/input/60k-data-with-context-v2/all_12_with_context2.csv")

In [11]:
train = pd.read_csv("/kaggle/input/kaggle-llm-science-exam/train.csv")

In [12]:
# import pickle
# with open("/kaggle/input/train-retrieved-articles/retrieved_articles.pkl", "rb") as f:
#     retrieved_articles = pickle.load(f)

In [13]:
# contexts = []

# for ra in retrieved_articles:
#     context = []
#     for el in ra[::-1][:4]:
#         context.append(el[2])
#     contexts.append("\n".join(context))
# print(len(contexts))

# train["context"] = contexts

In [14]:
# import pickle
# with open("/kaggle/input/train-60k-retrieved-articles-parsed/retrieved_articles_parsed.pkl", "rb") as f:
#     retrieved_articles_parsed = pickle.load(f)

In [15]:
# contexts = []

# for ra in retrieved_articles_parsed:
#     context = []
#     for el in ra[::-1][:4]:
#         context.append(el[2])
#     contexts.append("\n".join(context))
# print(len(contexts))

# train["context_2"] = contexts

In [16]:
# train.to_csv("train_context_270k.csv", index=False)

In [17]:
stop_words = ["don't",
 'did',
 'she',
 'shan',
 'am',
 'these',
 'isn',
 'use',
 'as',
 'but',
 'doesn',
 'will',
 'once',
 'after',
 "mustn't",
 'most',
 "isn't",
 'to',
 "that'll",
 'on',
 "needn't",
 'were',
 'other',
 'him',
 'some',
 'where',
 "you'd",
 'was',
 'of',
 'or',
 'any',
 'we',
 'my',
 'all',
 'under',
 'hasn',
 'you',
 "mightn't",
 "aren't",
 'his',
 'by',
 'just',
 'them',
 'wouldn',
 'itself',
 'up',
 "she's",
 'weren',
 'd',
 'here',
 'so',
 'while',
 'over',
 'off',
 'they',
 'each',
 'at',
 'own',
 'no',
 't',
 'this',
 'won',
 "shan't",
 'he',
 'having',
 'whom',
 'through',
 'too',
 'yourself',
 'between',
 'have',
 'll',
 'and',
 "you've",
 'until',
 'not',
 'what',
 'very',
 'has',
 'should',
 'yours',
 "weren't",
 'its',
 'is',
 'those',
 "you'll",
 'that',
 'down',
 'o',
 'ourselves',
 'needn',
 'their',
 'ours',
 'above',
 'ain',
 'into',
 "doesn't",
 'before',
 'herself',
 're',
 'out',
 've',
 'when',
 'than',
 "wasn't",
 'same',
 'i',
 'it',
 'used',
 'only',
 'does',
 'an',
 "wouldn't",
 'the',
 'can',
 'aren',
 'which',
 'do',
 'y',
 's',
 'there',
 'are',
 'be',
 'me',
 "didn't",
 'mustn',
 'our',
 'how',
 "hadn't",
 'who',
 'more',
 'against',
 "you're",
 'because',
 'yourselves',
 'if',
 "it's",
 "won't",
 'below',
 'hers',
 'a',
 "hasn't",
 'for',
 'didn',
 'both',
 'further',
 'been',
 'nor',
 'wasn',
 'shouldn',
 'himself',
 'such',
 'themselves',
 'haven',
 'again',
 'about',
 'during',
 'myself',
 'few',
 'ma',
 'theirs',
 'mightn',
 "haven't",
 'hadn',
 'had',
 'don',
 'then',
 'in',
 "shouldn't",
 'being',
 'm',
 'now',
 "should've",
 'doing',
 'her',
 'your',
 'from',
 'with',
 'couldn',
 "couldn't",
 'why']

In [18]:
def SplitList(mylist, chunk_size):
    return [mylist[offs:offs+chunk_size] for offs in range(0, len(mylist), chunk_size)]

def get_relevant_documents_parsed(df_valid):
    df_chunk_size=600
    paraphs_parsed_dataset = load_from_disk("/kaggle/working/all-paraphs-parsed-expanded")
    modified_texts = paraphs_parsed_dataset.map(lambda example:
                                             {'temp_text':
                                              f"{example['title']} {example['section']} {example['text']}".replace('\n'," ").replace("'","")},
                                             num_proc=2)["temp_text"]
    
    all_articles_indices = []
    all_articles_values = []
    for idx in tqdm(range(0, df_valid.shape[0], df_chunk_size)):
        df_valid_ = df_valid.iloc[idx: idx+df_chunk_size]
    
        articles_indices, merged_top_scores = retrieval(df_valid_, modified_texts)
        all_articles_indices.append(articles_indices)
        all_articles_values.append(merged_top_scores)
        
    article_indices_array =  np.concatenate(all_articles_indices, axis=0)
    articles_values_array = np.concatenate(all_articles_values, axis=0).reshape(-1)
    
    top_per_query = article_indices_array.shape[1]
    articles_flatten = [(
                         articles_values_array[index],
                         paraphs_parsed_dataset[idx.item()]["title"],
                         paraphs_parsed_dataset[idx.item()]["text"],
                        )
                        for index,idx in enumerate(article_indices_array.reshape(-1))]
    retrieved_articles = SplitList(articles_flatten, top_per_query)
    return retrieved_articles



def get_relevant_documents(df_valid):
    df_chunk_size=800
    
    cohere_dataset_filtered = load_from_disk("/kaggle/working/stem-wiki-cohere-no-emb")
    modified_texts = cohere_dataset_filtered.map(lambda example:
                                             {'temp_text':
                                              unicodedata.normalize("NFKD", f"{example['title']} {example['text']}").replace('"',"")},
                                             num_proc=2)["temp_text"]
    
    all_articles_indices = []
    all_articles_values = []
    for idx in tqdm(range(0, df_valid.shape[0], df_chunk_size)):
        df_valid_ = df_valid.iloc[idx: idx+df_chunk_size]
    
        articles_indices, merged_top_scores = retrieval(df_valid_, modified_texts)
        all_articles_indices.append(articles_indices)
        all_articles_values.append(merged_top_scores)
        
    article_indices_array =  np.concatenate(all_articles_indices, axis=0)
    articles_values_array = np.concatenate(all_articles_values, axis=0).reshape(-1)
    
    top_per_query = article_indices_array.shape[1]
    articles_flatten = [(
                         articles_values_array[index],
                         cohere_dataset_filtered[idx.item()]["title"],
                         unicodedata.normalize("NFKD", cohere_dataset_filtered[idx.item()]["text"]),
                        )
                        for index,idx in enumerate(article_indices_array.reshape(-1))]
    retrieved_articles = SplitList(articles_flatten, top_per_query)
    return retrieved_articles



def retrieval(df_valid, modified_texts):
    
    corpus_df_valid = df_valid.apply(lambda row:
                                     f'{row["prompt"]}\n{row["prompt"]}\n{row["prompt"]}\n{row["A"]}\n{row["B"]}\n{row["C"]}\n{row["D"]}\n{row["E"]}',
                                     axis=1).values
    vectorizer1 = TfidfVectorizer(ngram_range=(1,2),
                                 token_pattern=r"(?u)\b[\w/.-]+\b|!|/|\?|\"|\'",
                                 stop_words=stop_words)
    vectorizer1.fit(corpus_df_valid)
    vocab_df_valid = vectorizer1.get_feature_names_out()
    vectorizer = TfidfVectorizer(ngram_range=(1,2),
                                 token_pattern=r"(?u)\b[\w/.-]+\b|!|/|\?|\"|\'",
                                 stop_words=stop_words,
                                 vocabulary=vocab_df_valid)
    vectorizer.fit(modified_texts[:500000])
    corpus_tf_idf = vectorizer.transform(corpus_df_valid)
    
    print(f"length of vectorizer vocab is {len(vectorizer.get_feature_names_out())}")

    chunk_size = 100000
    top_per_chunk = 10
    top_per_query = 10

    all_chunk_top_indices = []
    all_chunk_top_values = []

    for idx in tqdm(range(0, len(modified_texts), chunk_size)):
        wiki_vectors = vectorizer.transform(modified_texts[idx: idx+chunk_size])
        temp_scores = (corpus_tf_idf * wiki_vectors.T).toarray()
        chunk_top_indices = temp_scores.argpartition(-top_per_chunk, axis=1)[:, -top_per_chunk:]
        chunk_top_values = temp_scores[np.arange(temp_scores.shape[0])[:, np.newaxis], chunk_top_indices]

        all_chunk_top_indices.append(chunk_top_indices + idx)
        all_chunk_top_values.append(chunk_top_values)

    top_indices_array = np.concatenate(all_chunk_top_indices, axis=1)
    top_values_array = np.concatenate(all_chunk_top_values, axis=1)
    
    merged_top_scores = np.sort(top_values_array, axis=1)[:,-top_per_query:]
    merged_top_indices = top_values_array.argsort(axis=1)[:,-top_per_query:]
    articles_indices = top_indices_array[np.arange(top_indices_array.shape[0])[:, np.newaxis], merged_top_indices]
    
    return articles_indices, merged_top_scores


In [19]:
retrieved_articles = get_relevant_documents(train)

Map (num_proc=2):   0%|          | 0/2781652 [00:00<?, ? examples/s]

  0%|          | 0/1 [00:00<?, ?it/s]/opt/conda/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:409: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ["'"] not in stop_words.
  warnings.warn(


length of vectorizer vocab is 10728



100%|██████████| 1/1 [04:45<00:00, 285.90s/it]


In [20]:
import pickle
with open("retrieved_articles.pkl", "wb") as f:
    pickle.dump(retrieved_articles, f)

In [21]:
retrieved_articles_parsed = get_relevant_documents_parsed(train)

Map (num_proc=2):   0%|          | 0/2101279 [00:00<?, ? examples/s]

  0%|          | 0/1 [00:00<?, ?it/s]/opt/conda/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:409: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ["'"] not in stop_words.
  warnings.warn(


length of vectorizer vocab is 10728



100%|██████████| 1/1 [05:54<00:00, 354.98s/it]


In [22]:
import pickle
with open("retrieved_articles_parsed.pkl", "wb") as f:
    pickle.dump(retrieved_articles_parsed, f)

In [23]:
contexts = []

for ra in retrieved_articles:
    context = []
    for el in ra[::-1][:4]:
        context.append(el[2])
    contexts.append("\n".join(context))
print(len(contexts))

train["context"] = contexts

200


In [24]:
contexts = []

for ra in retrieved_articles_parsed:
    context = []
    for el in ra[::-1][:4]:
        context.append(el[2])
    contexts.append("\n".join(context))
print(len(contexts))

train["context_2"] = contexts

200


In [25]:
train.to_csv("train_200_examples_context_270k.csv", index=False)